In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
import string
from nltk.corpus import stopwords
import re

#COLOCAR AQUI O CVS DE TRAINING
df = pd.read_csv('Data_Set/train.csv', header=0)
df = df.reset_index()  # make sure indexes pair with number of rows


In [3]:
#cleaning the data function ISTO DEMORA BUE


def clean_data(text,stops):
    text = text.lower()
    trans_table = str.maketrans({p: " " for p in string.punctuation})
    text = text.translate(trans_table)
    #precisamos de tirar dias da semana aqui-----------------------------------------------------------
    words = text.split()
    final = []
    for word in words:
         if word not in stops:
             final.append(word)
    final = " ".join(final)
    
    #forma exotica para remover pontuacao num texto
    #final = final.translate(str.maketrans("","",string.punctuation))
    
    # NOTA: caso os numeros nao tenham impacto na info dos textos
    # usamos isto para remover numeros (e talvez tenhamos de remover % tambem)
    # se os numeros forem importantes, talvez temos de modificar o texto de forma a indentificar os numeros
    # ex: 74,5% --> #PERCENT74,5
    
    final = "".join([i for i in final if not i.isdigit()])
    #forma NADA efeciente de tirar espaços
    while "  " in final:
        final = final.replace("  ", " ")
    return final
#the new data frame is going to have the same format but the text will be altered

base_weekdays = [
    "segunda-feira", "segundafeira", "segunda", "terça-feira", "terçafeira", "terça", "quarta-feira", "quartafeira", 
    "quarta", "quinta-feira", "quintafeira", "quinta", "sexta-feira",
    "sextafeira", "sexta", "sábado", "domingo"
]
meses_pt = [
    "janeiro",
    "fevereiro",
    "marco",
    "março",    # caso queira considerar com acento também
    "abril",
    "maio",
    "junho",
    "julho",
    "agosto",
    "setembro",
    "outubro",
    "novembro",
    "dezembro"
]
#ciclo iterrows
stops = stopwords.words("portuguese")
stops.extend(base_weekdays)

for index, row in df.iterrows():
    cleaned_data = clean_data(row['Text'],stops)

    df.at[index,'Text'] = cleaned_data

print(df)
    

       index                                               Text  \
0          0  pj operação internacional tráfico droga apreen...   
1          1  texto criado universidade república checa sist...   
2          2  nunca pague multas conduzir carta cartão cidad...   
3          3  antónio guterres estátua d afonso henriques in...   
4          4  mar droga gratuita cura menos doenças férias p...   
...      ...                                                ...   
63231  63231  governo diz estado vai indemnizar vítimas mort...   
63232  63232  homem cancro terminal realiza último desejo be...   
63233  63233  grupo impresa dívida milhões euros dívida novi...   
63234  63234  esa vai enviar sonda júpiter ciência tecnologi...   
63235  63235  mãe fica chocada perceber seguir filha superme...   

                                      Source  \
0                                         DN   
1                                        NaN   
2                            Contas Poupança   
3  

In [4]:
#Exemplo de data "limpa"
print(df.at[0,'Text'])

pj operação internacional tráfico droga apreendeu tonelada cocaína durante operação white sea iii vigiados movimentos embarcações realizadas ações busca inspeção feitas oito apreensões cocaína polícia judiciária pj participou operação internacional combate tráfico droga via marítima terminou detenção presumíveis traficantes apreensão cerca toneladas cocaína acordo comunicado divulgado feira direção nacional pj operação white sea iii decorreu junho julho durante vigiados movimentos embarcações realizadas ações busca inspeção feitas oito apreensões cocaína operação focou vários modi operandi organizações criminosas utilizam introduzir droga europa incidiu sobre movimentos embarcações determinados locais costa atlântica continente europeu canal mancha mar norte bem atividades desenvolvidas alguns portos marítimos operação incluída planos ação european multidisciplinary platform against criminal threats empcat união europeia contou participação autoridades alemanha bélgica dinamarca espanh

In [ ]:
# TF IDF podemos/devemos alterar o ngram_range e podemos colocar mais argumentos
vectorizer = TfidfVectorizer( lowercase=True,
                              max_features=5000,
                              max_df = 0.9,
                              min_df= 10,
                              ngram_range= (1,3),
                              stop_words = "english",
                              use_idf=True,
                              smooth_idf=True
                            )

vectors = vectorizer.fit_transform(df['Text'])

feature_names = vectorizer.get_feature_names_out()

dense = vectors.todense()
denselist = dense.tolist()

all_keywords = []

for description in denselist:
    x=0
    keywords = []
    for word in description:
        if word > 0:
            keywords.append(feature_names[x])
        x=x+1
    all_keywords.append(keywords)
 
print(all_keywords[0])

In [ ]:
def print_tfidf_summary(vectors):
    n_docs, n_feats = vectors.shape
    nnz = vectors.nnz
    density = nnz / (n_docs * n_feats)
    print(f"shape: {n_docs} x {n_feats}")
    print(f"non-zeros: {nnz}  density: {density:.4%}")

def top_features_for_doc(vectors, feature_names, doc_idx=0, top_n=10):
    row = vectors.getrow(doc_idx).tocoo()
    if row.nnz == 0:
        print(f"doc {doc_idx}: no non-zero features")
        return
    idx = np.argsort(row.data)[::-1][:top_n]
    terms = [feature_names[row.col[i]] for i in idx]
    scores = row.data[idx]
    print(f"Top {top_n} features for doc {doc_idx}:")
    for t, s in zip(terms, scores):
        print(f"  {t}: {s:.4f}")

def top_features_global(vectors, feature_names, top_n=30, by="mean"):
    # by="mean" or "sum"
    if by == "mean":
        vals = np.asarray(vectors.mean(axis=0)).ravel()
    else:
        vals = np.asarray(vectors.sum(axis=0)).ravel()
    top_idx = np.argsort(vals)[::-1][:top_n]
    print(f"Top {top_n} features by {by}:")
    for i in top_idx:
        print(f"  {feature_names[i]}: {vals[i]:.6f}")

def as_dataframe(vectors, feature_names, max_cells=1e9):
    # Convert only if not too large
    n_docs, n_feats = vectors.shape
    if n_docs * n_feats > max_cells:
        raise MemoryError("matrix too large to convert to dense DataFrame")
    return pd.DataFrame(vectors.toarray(), columns=feature_names)

# usage examples
print_tfidf_summary(vectors)
top_features_for_doc(vectors, feature_names, doc_idx=0, top_n=15)
top_features_global(vectors, feature_names, top_n=40, by="mean")

In [ ]:
# --- Corpus-level term summary and prints ---
# compute sums, means and document frequencies for each feature
sums = np.asarray(vectors.sum(axis=0)).ravel()
means = np.asarray(vectors.mean(axis=0)).ravel()
df_counts = np.diff(vectors.tocsc().indptr)  # documents-per-term

summary = pd.DataFrame({
    'term': feature_names,
    'sum': sums,
    'mean': means,
    'df': df_counts
})
if hasattr(vectorizer, 'idf_'):
    summary['idf'] = vectorizer.idf_

# print top terms by different metrics
def print_top_terms(summary, by='sum', top_n=30):
    print(f"\nTop {top_n} terms by {by}:")
    top = summary.sort_values(by=by, ascending=False).head(top_n)
    print(top[['term', 'sum', 'mean', 'df'] + (['idf'] if 'idf' in top.columns else [])].to_string(index=False))

print_top_terms(summary, by='sum', top_n=30)
print_top_terms(summary, by='df', top_n=30)

# show sample cleaned texts for first 5 documents
print('\nSample cleaned texts (first 5 rows):')
for i in range(min(5, len(df))):
    print(f"[{i}] ", df['Text'].iloc[i])

# save summary CSV
summary_out = 'tfidf_term_summary.csv'
summary.sort_values(by='sum', ascending=False).to_csv(summary_out, index=False)
print(f"\nSaved term summary to: {summary_out}")

In [ ]:
import os
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

X = vectors  # sparse TF-IDF matrix 
y = df['Label'].values  

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = 'f1_macro'
results_dir = 'cv_results'
os.makedirs(results_dir, exist_ok=True)

In [ ]:
models = {
    'decision_tree': {
        'pipeline': Pipeline([('clf', DecisionTreeClassifier(random_state=42))]),
        'params': {
            'clf__criterion': ['gini', 'entropy'],
            'clf__max_depth': [None, 10, 20],
            'clf__min_samples_leaf': [1, 2, 5]
        }
    },
    'gaussian_nb': {
        # use SVD to reduce dimensionality and produce dense input for GaussianNB
        'pipeline': Pipeline([('svd', TruncatedSVD(n_components=200, random_state=42)),
                              ('clf', GaussianNB())]),
        'params': {
            'svd__n_components': [100, 200],
            'clf__var_smoothing': [1e-9, 1e-8]
        }
    },
    'logreg_l2': {
        'pipeline': Pipeline([('clf', LogisticRegression(penalty='l2', solver='saga', max_iter=2000, random_state=42))]),
        'params': {
            'clf__C': [0.01, 0.1, 1, 10],
            'clf__class_weight': [None, 'balanced']
        }
    },
    'logreg_l1': {
        'pipeline': Pipeline([('clf', LogisticRegression(penalty='l1', solver='saga', max_iter=2000, random_state=42))]),
        'params': {
            'clf__C': [0.01, 0.1, 1, 10],
            'clf__class_weight': [None, 'balanced']
        }
    },
    'mlp': {
        # SVD -> scaler -> MLP
        'pipeline': Pipeline([
            ('svd', TruncatedSVD(n_components=200, random_state=42)),
            ('scaler', StandardScaler()),
            ('clf', MLPClassifier(max_iter=800, random_state=42))
        ]),
        'params': {
            'svd__n_components': [100, 200],
            'clf__hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
            'clf__alpha': [1e-4, 1e-3],
            'clf__learning_rate_init': [1e-3],
            'clf__activation': ['relu']
        }
    }
}

In [ ]:
# Run grid searches
for name, spec in models.items():
    print(f"\n=== GridSearchCV: {name} ===")
    gs = GridSearchCV(spec['pipeline'], spec['params'],
                      cv=cv, scoring=scoring, n_jobs=-1, verbose=0, refit=True)
    gs.fit(X, y)
    print("best params:", gs.best_params_)
    # cross-validated predictions using the best estimator
    y_pred = cross_val_predict(gs.best_estimator_, X, y, cv=cv, n_jobs=-1)
    print("classification report (cv):")
    print(classification_report(y, y_pred, digits=4))
    # save results
    pd.DataFrame(gs.cv_results_).to_csv(os.path.join(results_dir, f"{name}_cv_results.csv"), index=False)
    joblib.dump(gs.best_estimator_, os.path.join(results_dir, f"{name}_best_estimator.joblib"))
    print(f"Saved cv_results and best_estimator for {name} in {results_dir}")

print("\nAll searches complete. Results directory:", results_dir)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

RESULTS_DIR = "cv_results"
# COLOCAR AQUI O CVS DE VALIDACAO
DATA_CSV = "Data_Set/val.csv"
X_SPARSE_PATH = os.path.join(RESULTS_DIR, "X_sparse.joblib")
VEC_PATH = "tfidf_vectorizer.joblib"
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
OUT_METRICS = os.path.join(RESULTS_DIR, "metrics_comparison.csv")
OUT_ROC = os.path.join(RESULTS_DIR, "best_model_roc.png")

def load_features():
    if os.path.exists(X_SPARSE_PATH):
        X = joblib.load(X_SPARSE_PATH)
        print(f"Loaded X sparse from {X_SPARSE_PATH}")
        return X
    if os.path.exists(VEC_PATH):
        vec = joblib.load(VEC_PATH)
        df = pd.read_csv(DATA_CSV)
        X = vec.transform(df['Text'].fillna('').astype(str))
        print(f"Transformed texts with loaded vectorizer {VEC_PATH}")
        return X
    raise FileNotFoundError("No X_sparse.joblib or tfidf_vectorizer.joblib found; run cv_search first.")

def load_data():
    df = pd.read_csv(DATA_CSV)
    if 'Label' not in df.columns:
        raise RuntimeError("CSV must contain 'Label' column")
    y = df['Label'].values
    return df, y

def load_estimators():
    files = [f for f in os.listdir(RESULTS_DIR) if f.endswith("_best_estimator.joblib")]
    files.sort()
    if not files:
        raise RuntimeError(f"No *_best_estimator.joblib files found in {RESULTS_DIR}")
    estimators = {}
    for f in files:
        name = f.replace("_best_estimator.joblib", "")
        estimators[name] = joblib.load(os.path.join(RESULTS_DIR, f))
    return estimators

def try_get_scores(est, X, y):
    # attempt predict_proba then decision_function; return None if not available
    try:
        probs = cross_val_predict(est, X, y, cv=CV, method='predict_proba', n_jobs=-1)
        return probs
    except Exception:
        try:
            scores = cross_val_predict(est, X, y, cv=CV, method='decision_function', n_jobs=-1)
            return scores
        except Exception:
            return None

def compute_metrics(estimators, X, y):
    rows = []
    scores_cache = {}
    for name, est in estimators.items():
        print(f"Evaluating: {name}")
        y_pred = cross_val_predict(est, X, y, cv=CV, n_jobs=-1)
        acc = accuracy_score(y, y_pred)
        prec = precision_score(y, y_pred, average='macro', zero_division=0)
        rec = recall_score(y, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y, y_pred, average='macro', zero_division=0)
        rows.append({
            'model': name,
            'accuracy': acc,
            'precision_macro': prec,
            'recall_macro': rec,
            'f1_macro': f1
        })
        scores = try_get_scores(est, X, y)
        if scores is not None:
            scores_cache[name] = scores
    dfm = pd.DataFrame(rows).sort_values(by='f1_macro', ascending=False).reset_index(drop=True)
    dfm.to_csv(OUT_METRICS, index=False)
    print(f"\nSaved metrics table to {OUT_METRICS}\n")
    print(dfm.to_string(index=False, float_format="%.4f"))
    return dfm, scores_cache

def plot_roc_for_best(dfm, scores_cache, y):
    best_name = dfm.loc[0, 'model']
    print(f"\nBest model by f1_macro: {best_name}")
    if best_name not in scores_cache:
        print("No probability/score available for best model; cannot plot ROC/AUC.")
        return
    y_score = scores_cache[best_name]
    classes = np.unique(y)
    n_classes = len(classes)
    plt.figure(figsize=(8,6))
    if n_classes == 2:
        # binary: ensure shape and get positive-class scores
        if y_score.ndim == 2 and y_score.shape[1] >= 2:
            prob_pos = y_score[:,1]
        else:
            prob_pos = y_score.ravel()
        fpr, tpr, _ = roc_curve(y, prob_pos, pos_label=classes[-1])
        auc_val = roc_auc_score(y, prob_pos)
        plt.plot(fpr, tpr, label=f'{best_name} (AUC = {auc_val:.4f})', lw=2)
    else:
        # multiclass: one-vs-rest per-class curves + macro AUC
        y_bin = label_binarize(y, classes=classes)
        aucs = []
        for i, cls in enumerate(classes):
            if y_score.ndim == 2 and y_score.shape[1] == n_classes:
                score_i = y_score[:, i]
            else:
                score_i = y_score.ravel()
            fpr, tpr, _ = roc_curve(y_bin[:, i], score_i)
            auc_i = roc_auc_score(y_bin[:, i], score_i)
            aucs.append(auc_i)
            plt.plot(fpr, tpr, lw=1, label=f'class {cls} (AUC={auc_i:.3f})')
        try:
            macro_auc = roc_auc_score(y_bin, y_score, average='macro', multi_class='ovr')
        except Exception:
            macro_auc = np.mean(aucs)
        plt.plot([], [], ' ', label=f'macro AUC = {macro_auc:.4f}')
    plt.plot([0,1], [0,1], 'k--', alpha=0.5)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC curve — {best_name}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(OUT_ROC)
    print(f"Saved ROC plot to {OUT_ROC}")
    plt.show()

In [ ]:
def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df, y = load_data()
    X = load_features()
    estimators = load_estimators()
    dfm, scores_cache = compute_metrics(estimators, X, y)
    plot_roc_for_best(dfm, scores_cache, y)

if __name__ == "__main__":
    main()